# 04.3 — Fine-tuning Qwen3-0.6B con LoRA

Este cuaderno añade un tercer modelo de lenguaje **después** de completar `04_2`. Usa exactamente el dataset 4:1 y las particiones guardadas por aquel cuaderno, entrena únicamente las cinco categorías gruesas y compara Qwen con el mejor clásico, MiniLM y E5. No entrena etiquetas finas ni flags transversales.

Qwen se mantiene separado porque su ajuste es más costoso y puede ejecutarse en CPU local, Google Colab con GPU o un servidor Jupyter remoto conectado desde VS Code. Los resultados se escriben como JSON y en `resultados/INFORME_FINETUNING_QWEN3_LORA.md`; no se generan CSV de Qwen.

## 0. Entorno local, Colab o servidor remoto

La opción preferida es una GPU CUDA. Para Colab alojado, copie o sincronice **todo el proyecto** en Google Drive, monte Drive y ejecute el cuaderno desde la interfaz web de Colab. Como alternativa, VS Code puede conectarse a una URL autenticada de un servidor Jupyter remoto; no se presupone conexión directa de VS Code al kernel alojado de Colab. En ambos casos el directorio remoto debe conservar la misma estructura. Los artefactos del `04_2` no se reconstruyen ni se descargan de manera aislada: sus hashes se verifican antes de entrenar (Google, s. f.; Microsoft, s. f.).

In [ ]:
%pip install -q "torch>=2.6,<3" "transformers>=5,<6" "peft>=0.18,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.9,<4" "tqdm>=4.67,<5" "joblib>=1.4,<2"

In [ ]:
from pathlib import Path
import importlib
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display, Markdown

# En Colab, descomente y ajuste si el proyecto está en Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT_OVERRIDE = Path('/content/drive/MyDrive/Trabajo_PLN-MIA-Grupo4')
PROJECT_ROOT_OVERRIDE = None

search_start = PROJECT_ROOT_OVERRIDE or Path.cwd().resolve()
for candidate in (search_start, *search_start.parents):
    if (candidate / 'scripts_auxiliares' / 'entrenar_transformers_gruesos.py').exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Ajuste PROJECT_ROOT_OVERRIDE a la copia completa del proyecto.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts_auxiliares import entrenar_transformers_gruesos as tm
tm = importlib.reload(tm)
tm.set_reproducibility()
print('Raíz:', ROOT)
print('Dispositivo Qwen:', tm.qwen_device())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU: el entrenamiento es viable en memoria, pero puede tomar muchas horas.')

## 1. Cargar y verificar resultados del 04.2

Esta es una dependencia fuerte. Se verifican el SHA-256 del dataset 4:1 y los artefactos guardados del clásico, MiniLM y E5. Si falta alguno, vuelva al `04_2` y ejecute su comparación final.

In [ ]:
frames, audit = tm.load_experiment_frames()
registry_04_2 = tm.load_model_registry()
required = {
    'paraphrase_minilm', 'e5_small'
}
available = {item['model_key'] for item in registry_04_2['models']}
if not required <= available or not any(key.startswith('classical__') for key in available):
    raise RuntimeError('El registro del 04_2 no contiene clásico, MiniLM y E5.')

analysis_classical = json.loads((tm.METRICS_DIR / 'operacion_mejor_clasico.json').read_text(encoding='utf-8'))
analysis_minilm = json.loads((tm.METRICS_DIR / 'operacion_paraphrase_minilm.json').read_text(encoding='utf-8'))
analysis_e5 = json.loads((tm.METRICS_DIR / 'operacion_e5_small.json').read_text(encoding='utf-8'))
display(tm.operational_comparison_frame([analysis_classical, analysis_minilm, analysis_e5]))
display(tm.registered_artifact_frame(registry_04_2))
print('Dataset verificado:', registry_04_2['dataset'])
print('SHA-256:', registry_04_2['dataset_sha256'])

## 2. Por qué Qwen3-0.6B-Base + LoRA

GPT-3 no es un checkpoint abierto que pueda descargarse y ajustarse localmente. La alternativa abierta actual de OpenAI es `gpt-oss-20b`, pero su guía oficial de fine-tuning está diseñada para una H100 de 80 GB. DeepSeek-V2-Lite y los checkpoints generales Kimi/Moonlight más pequeños parten de 16B parámetros. Frente a ellos, `Qwen/Qwen3-0.6B-Base` tiene licencia Apache-2.0, 0,6B parámetros, preentrenamiento declarado en 119 idiomas y soporte `Qwen3ForSequenceClassification`; es la comparación generativa proporcionada al hardware disponible (DeepSeek-AI, 2024; Moonshot AI, s. f.; OpenAI, s. f.; Qwen Team, 2025).

No se hace fine-tuning completo. LoRA congela el modelo base e introduce matrices entrenables de bajo rango en las proyecciones Q/K/V/O (Hu et al., 2022). Se usa rango 8, alpha 16, dropout 0,05, batch físico 2, acumulación 4, 128 tokens y máximo dos épocas. Son parámetros fijados antes de consultar el test.

In [ ]:
display(pd.DataFrame({
    'parámetro': ['checkpoint', 'revisión', 'train', 'validación', 'test', 'batch físico', 'acumulación', 'batch efectivo', 'épocas máximas', 'learning rate', 'LoRA rank', 'longitud'],
    'valor': [tm.QWEN_LORA_SPEC.model_id, tm.QWEN_LORA_SPEC.revision, len(frames['train']), len(frames['validation']), len(frames['test']), tm.QWEN_TRAIN_BATCH_SIZE, tm.QWEN_GRADIENT_ACCUMULATION, tm.QWEN_TRAIN_BATCH_SIZE * tm.QWEN_GRADIENT_ACCUMULATION, tm.QWEN_MAX_EPOCHS, tm.QWEN_LEARNING_RATE, tm.QWEN_LORA_RANK, tm.MAX_LENGTH],
}))
print('La barra de la siguiente celda mostrará lotes, pérdida, learning rate y ETA real.')

## 3. Fine-tuning LoRA de Qwen

Ejecute esta celda una sola vez. Usa los 14.064 chunks de train; no toma filas de validación o test. Cada época termina con métricas de validación y un adaptador `last_adapter`; cuando mejora PR-AUC se actualiza `best_adapter`. En CPU puede tardar muchas horas; una GPU CUDA de Colab o servidor es preferible.

In [ ]:
training_qwen = tm.run_qwen_lora_finetuning(frames)
display(pd.DataFrame(training_qwen['history'])[[
    'epoch', 'training_loss', 'damage_pr_auc_macro',
    'damage_f1_macro', 'damage_recall_micro', 'epoch_seconds'
]])
print('Mejor época:', training_qwen['best_epoch'])
print('Tiempo total (h):', training_qwen['training_seconds'] / 3600)
print('Adaptador:', training_qwen['adapter'])

## 4. Evaluación y puerta operativa de Qwen

El adaptador de la mejor época se evalúa con los mismos umbrales metodológicos del `04_2`. El umbral de alerta se calibra en validación para 95 % de recall y se congela antes del test.

In [ ]:
evaluation_qwen = tm.evaluate_qwen_lora_model(frames)
analysis_qwen = tm.analyze_qwen_operational(frames, evaluation_qwen)
analyses_all = [analysis_classical, analysis_minilm, analysis_e5, analysis_qwen]
display(tm.operational_comparison_frame(analyses_all))
display(pd.DataFrame(analysis_qwen['autonomous']['test']['category_recall'].items(), columns=['categoría', 'recall Qwen test']))
display(pd.DataFrame([
    analysis_qwen['human_review_alert']['validation'],
    analysis_qwen['human_review_alert']['test'],
], index=['validación', 'test'])[['review_rate', 'recall', 'recall_wilson_95', 'negative_predictive_value', 'false_negatives']])

## 5. Comparación final, modelo guardado y filosofía de producción

Se elige entre los candidatos que superen la misma puerta operativa. Dentro del mismo modo se usa PR-AUC de validación; el test confirma aceptación y no ordena modelos. Las configuraciones de MiniLM, E5 y Qwen quedaron fijadas antes de consultar el test correspondiente y no deben reajustarse tras verlo. La moderación autónoma exige además gold standard humano independiente, prevalencia natural y piloto prospectivo.

In [ ]:
final_comparison = tm.operational_comparison_frame(analyses_all)
final_selection = tm.select_production_candidate(analyses_all)
display(final_comparison)
display(Markdown(
    f"### Decisión final\n\n**Modelo:** {final_selection.get('selected_model_label') or 'ninguno'}  \
"
    f"**Modo:** `{final_selection['operating_mode']}`  \
"
    f"**Estado:** `{final_selection['status']}`  \
"
    f"**Autonomía:** {final_selection['autonomous_deployment_supported']}  \
"
    f"**Alerta humana:** {final_selection['human_review_alert_supported']}"
))

classical = json.loads((tm.METRICS_DIR / 'comparacion_modelos_clasicos.json').read_text(encoding='utf-8'))
trainings_04_2 = {
    key: json.loads((tm.METRICS_DIR / f'finetuning_{key}.json').read_text(encoding='utf-8'))
    for key in tm.MODEL_SPECS
}
evaluations_04_2 = {
    key: json.loads((tm.METRICS_DIR / f'evaluacion_{key}.json').read_text(encoding='utf-8'))
    for key in tm.MODEL_SPECS
}
final_registry = tm.write_model_registry(
    classical, trainings_04_2, evaluations_04_2, final_selection,
    qwen_training=training_qwen, qwen_evaluation=evaluation_qwen,
)
tm.write_operational_decision_report(analyses_all, final_selection)
tm.write_qwen_report(
    training_qwen, evaluation_qwen, analyses_all, final_selection, final_registry
)
display(tm.registered_artifact_frame(final_registry))
print('Registro actualizado:', tm.MODEL_REGISTRY_PATH.relative_to(ROOT))
print('Informe Qwen:', tm.QWEN_REPORT_PATH.relative_to(ROOT))
print("Recarga opcional: modelo, metadatos = tm.load_registered_model(final_selection['selected_model_key'])")

In [ ]:
plot_frame = final_comparison.set_index('modelo')[[
    'PR-AUC daño validación', 'PR-AUC daño test',
    'tasa revisión humana', 'recall alerta test'
]]
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
plot_frame[['PR-AUC daño validación', 'PR-AUC daño test']].plot.bar(ax=axes[0], color=['#4C78A8', '#F58518'])
plot_frame[['tasa revisión humana', 'recall alerta test']].plot.bar(ax=axes[1], color=['#E45756', '#54A24B'])
axes[0].set_title('Comparación de ranking')
axes[1].set_title('Costo–recall de revisión humana')
for ax in axes:
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=.25)
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
figure_path = tm.FIGURES_DIR / 'comparacion_final_con_qwen.png'
plt.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Figura:', figure_path.relative_to(ROOT))

## Referencias (APA 7)

DeepSeek-AI. (2024). *DeepSeek-V2-Lite* [Modelo de lenguaje]. Hugging Face. https://huggingface.co/deepseek-ai/DeepSeek-V2-Lite

Geifman, Y., & El-Yaniv, R. (2017). Selective classification for deep neural networks. In *Advances in Neural Information Processing Systems* (Vol. 30). https://proceedings.neurips.cc/paper/2017/hash/4a8423d5e91fda00bb7e46540e2b0cf1-Abstract.html

Google. (s. f.). *Colaboratory: Local runtimes*. https://research.google.com/colaboratory/local-runtimes.html

Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., & Chen, W. (2022). LoRA: Low-rank adaptation of large language models. In *International Conference on Learning Representations*. https://openreview.net/forum?id=nZeVKeeFYf9

Microsoft. (s. f.). *Jupyter Notebooks in Visual Studio Code*. https://code.visualstudio.com/docs/datascience/jupyter-notebooks

Moonshot AI. (s. f.). *Modelos publicados por Moonshot AI* [Colección de modelos]. Hugging Face. https://huggingface.co/moonshotai/models

OpenAI. (s. f.). *Fine-tuning a multilingual reasoner with Hugging Face*. https://developers.openai.com/cookbook/articles/gpt-oss/fine-tune-transfomers

Qwen Team. (2025). *Qwen3 technical report*. arXiv. https://doi.org/10.48550/arXiv.2505.09388

Qwen Team. (2025). *Qwen3-0.6B-Base* [Modelo de lenguaje]. Hugging Face. https://huggingface.co/Qwen/Qwen3-0.6B-Base

Tonneau, M., Quinta de Castro, P. V., Lasri, K., Farouq, I., Subramanian, L., Orozco-Olvera, V., & Fraiberger, S. P. (2024). NAIJAHATE: Evaluating hate speech detection on Nigerian Twitter using representative data. In *Proceedings of the 62nd Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers)* (pp. 9020–9040). Association for Computational Linguistics. https://aclanthology.org/2024.acl-long.488/